In [5]:
"""Professional BERT Text Classification - Google Colab"""

# ======================
# 1. INSTALL & IMPORT
# ======================
!pip install transformers datasets accelerate torch scikit-learn huggingface_hub -q

import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import re

print(" Libraries installed and imported")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 29.0 MB/s eta 0:00:00
 Libraries installed and imported


In [6]:
# ======================
# 2.1 MOUNT GOOGLE DRIVE
# ======================
from google.colab import drive

# ============================================
# 2.2 Prepare Drive Folder
# ============================================
import os
drive_folder = '.'
os.makedirs(drive_folder, exist_ok=True)
print(f" Drive folder ready: {drive_folder}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Drive folder ready: /content/drive/MyDrive/research_papers_Classification


In [7]:
# ======================
# 3. LOAD & PREPROCESS DATA
# ======================
def load_and_clean_data(file_path):
    """Load and clean the dataset efficiently"""
    df = pd.read_csv(file_path, encoding='latin1')
    df = df[['main_category', 'full_abstract']].copy()

    # Clean text
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text)
        return text.lower().strip()

    df['cleaned_text'] = df['full_abstract'].apply(clean_text)
    df['main_category'] = df['main_category'].apply(clean_text)

    # Remove empty and duplicates
    df = df.dropna(subset=['main_category', 'cleaned_text'])
    df = df.drop_duplicates(subset=['cleaned_text'])

    return df

# Load your data - UPDATE THIS PATH
data_path = "./cleaned_dataset.csv"  # ← Update this path
df = load_and_clean_data(data_path)

print(f" Dataset loaded: {len(df)} samples")
print(f" Categories: {df['main_category'].value_counts().to_dict()}")


 Dataset loaded: 139732 samples
 Categories: {'medicine': 16938, 'psychology': 16724, 'chemistry': 16652, 'business': 15999, 'physics': 15851, 'environmentalscience': 15767, 'mathematics': 15448, 'computerscience': 14993, 'biology': 11360}


In [8]:

# ======================
# 4. ENCODE LABELS
# ======================
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['main_category'])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

print(" Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"   {i}: {label}")

 Label mapping:
   0: biology
   1: business
   2: chemistry
   3: computerscience
   4: environmentalscience
   5: mathematics
   6: medicine
   7: physics
   8: psychology


In [9]:

# ======================
# 5. TRAIN-TEST SPLIT
# ======================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f" Train: {len(train_df)}, Test: {len(test_df)}")


 Train: 111785, Test: 27947


In [10]:

# ======================
# 6. BERT TOKENIZATION
# ======================
model_name = "bert-base-uncased"  # You can try "distilbert-base-uncased" for faster training
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["cleaned_text"],
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['cleaned_text', 'label']])

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print(" Tokenization completed")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/111785 [00:00<?, ? examples/s]

Map:   0%|          | 0/27947 [00:00<?, ? examples/s]

 Tokenization completed


## You can jump from here to try the latest trained version.

In [12]:

# ======================
# 7. LOAD BERT MODEL
# ======================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

print(f" Model loaded with {len(label_encoder.classes_)} classes")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Model loaded with 9 classes


In [ ]:

# ======================
# 8. TRAINING SETUP (FIXED)
# ======================
# Define compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# FIXED Training arguments for latest transformers version
training_args = TrainingArguments(
    output_dir="./bert_classification_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2, # 3
    weight_decay=0.01,

    # FIXED: New parameter names
    eval_strategy="epoch",          # كان evaluation_strategy
    save_strategy="epoch",          # كان save_strategy
    logging_strategy="epoch",       # جديد مطلوب
    save_steps=500,
    eval_steps=500,
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="./bert_classification_logs",
    save_total_limit=2,
    report_to=None,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(" Starting training...")


In [ ]:


# ======================
# 9. TRAIN MODEL
# ======================
trainer.train()

print(" Training completed!")


In [ ]:

# ======================
# 10. EVALUATE MODEL
# ======================
results = trainer.evaluate()
print(f"Final evaluation results: {results}")

# Detailed classification report
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\n Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


In [ ]:

# ======================
# 11. SAVE MODEL
# ======================
# Save to Google Drive
save_path = "./bert_text_classifier"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

from google.colab import userdata
token = userdata.get('hf_token')

model_repo_name = "Emran025/bert-text-classification"  # غير your-username
trainer.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)
print(f"Model uploaded to: https://huggingface.co/{model_repo_name}")

In [ ]:

# ======================
# 12. INFERENCE FUNCTION
# ======================
def predict_text(text, model_path=save_path):
    """Predict category for new text"""
    # Load model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)

    # Get probabilities
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_class].item()

    # Get label mapping from saved config
    id2label = model.config.id2label

    return id2label[pred_class], confidence


In [ ]:

# ======================
# 13. TEST PREDICTIONS
# ======================
print("\n Testing predictions:")

# Test samples from the dataset
test_samples = test_df.head(3)
for idx, row in test_samples.iterrows():
    true_label = row['main_category']
    text = row['cleaned_text'][:300] + "..."  # First 300 chars

    pred_label, confidence = predict_text(text)

    print(f"\n Sample {idx}:")
    print(f"   Text: {text}")
    print(f"   True: {true_label}")
    print(f"   Pred: {pred_label} ({confidence:.2%})")
    print(f"   Status: {'[True]' if pred_label == true_label else '[False]'}")


# Access the latest version of training data

In [23]:
from transformers import AutoModelForSequenceClassification, Trainer

import os

# المسار الذي حفظت فيه نتائج التدريب
output_dir = "./bert_classification_output"

# العثور على آخر checkpoint
def get_last_checkpoint(dir_path):
    checkpoints = [os.path.join(dir_path, d) for d in os.listdir(dir_path) if d.startswith("checkpoint")]
    if not checkpoints:
        return None
    checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))
    return checkpoints[-1]

last_checkpoint = get_last_checkpoint(output_dir)
print(" Using model from:", last_checkpoint)


model = AutoModelForSequenceClassification.from_pretrained(last_checkpoint)


 Using model from: /content/drive/MyDrive/research_papers_Classification/bert_classification_output/checkpoint-13976


In [24]:
from transformers import Trainer

# إنشاء الـ Trainer لتقييم النموذج حتى بعد انتهاء التحميل
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    eval_dataset=tokenized_test,  # بيانات الاختبار أو التقييم
    compute_metrics=lambda p: {"accuracy": (p.predictions.argmax(-1) == p.label_ids).mean()},
)


/tmp/ipython-input-2080366856.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [28]:
from transformers import pipeline

# إنشاء مصنف نصوص جاهز للاستخدام
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# مثال على اختبار نص جديد بعنوان Mental health من Mendeley
example = "Mental health has long been defined as the absence of psychopathologies, such as depression and anxiety. The absence of mental illness, however, is a minimal outcome from a psychological perspective on lifespan development. This article therefore focuses on mental illness as well as on three core components of positive mental health: feelings of happiness and satisfaction with life (emotional well-being), positive individual functioning in terms of self-realization (psychological well-being), and positive societal functioning in terms of being of social value (social well-being). The two continua model holds that mental illness and mental health are related but distinct dimensions. This model was studied on the basis of a cross-sectional representative internet survey of Dutch adults (N = 1,340; 18-87 years). Mental illness was measured with the Brief Symptom Inventory and mental health with the Mental Health Continuum Short Form. It was found that older adults, except for the oldest-old, scored lower on psychopathological symptoms and were less likely to be mentally ill than younger adults. Although there were fewer age differences for mental health, older adults experienced more emotional, similar social and slightly lower psychological well-being. In sum, today's older adults have fewer mental illness problems, but they are not in a better positive mental health than today's younger adults. These findings support the validity of the two continua model in adult development. © 2009 The Author(s)."
result = classifier(example)
print(result)
# مثال على اختبار نص جديد بعنوان Fever من Mendeley
example = "Fever of unknown origin (FUO) in Pediatrics can be defined as an entity in which fever is the main sign, without having reached an etiological diagnosis after more than 7 days, despite the correct performance of the history, physical examination, and first-level complementary tests. In most cases, the cause is an uncommon manifestation of a common disease, especially an infectious disease. Other less frequent causes are inflammatory or autoimmune, followed by a miscellaneous of diseases and finally, malignant diseases. In a significant percentage of cases, a diagnosis is not reached, being this a good predictor of spontaneous resolution without sequelae. Thorough and repeated history-taking and physical examination throughout the evaluation is the essential element in the diagnostic process of FUO. It will allow us to reach the cause or optimize the complementary tests that will lead us to the cause with the least harm to the patient and in an efficient manner. Empirical antibiotic or anti-inflammatory treatment should be avoided in stable patients without a specific diagnostic suspicion, as they could mask findings that would bring us closer to the diagnosis."
result = classifier(example)
print(result)
# مثال على اختبار نص جديد بعنوان AI من Mendeley
example = "Today, intelligent systems that offer artificial intelligence capabilities often rely on machine learning. Machine learning describes the capacity of systems to learn from problem-specific training data to automate the process of analytical model building and solve associated tasks. Deep learning is a machine learning concept based on artificial neural networks. For many applications, deep learning models outperform shallow machine learning models and traditional data analysis approaches. In this article, we summarize the fundamentals of machine learning and deep learning to generate a broader understanding of the methodical underpinning of current intelligent systems. In particular, we provide a conceptual distinction between relevant terms and concepts, explain the process of automated analytical model building through machine learning and deep learning, and discuss the challenges that arise when implementing such intelligent systems in the field of electronic markets and networked business. These naturally go beyond technological aspects and highlight issues in human-machine interaction and artificial intelligence servitization."
result = classifier(example)
print(result)
# مثال على اختبار نص جديد
example = "FCC gasoline is a major component in the total gasoline pool produced in an integrated refinery, but it contains many compounds (olefins, sulfur and aromatics) which lead to harmful automobile emissions. The objective of the present study is to determine the effect of feedstock quality on gasoline composition in a range of operating variables with a constant type of catalyst. The work was carried out in an FCC pilot plant constructed and operated in CPERI. The FCC gasoline was fully analyzed in a system of GC/MS. Ten different feedstocks were used in the unit in order to investigate the feedstock physical properties which affect the gasoline yield and composition, the feed conversion and the coke yield as well. The gasoline components were measured as total hydrocarbon groups: aromatics, normal and branched olefins, normal and isoparaffins and naphthenes but special emphasis was given, in this study, for the aromatic and olefinic content of gasoline. The main conclusion of the work is that feed conversion, coke yield and gasoline yield and composition are strongly influenced by the type of FCC feedstock. It was shown that a paraffinic and an aromatic FCC feedstock produce, respectively, an olefinic or an aromatic gasoline. The hydrotreating process plays also an important role in the gasoline composition. For these feed effects detailed qualitative and quantitative information is given in the paper. Moreover, short form models were proposed for the prediction of conversion coke yield and gasoline composition as a function of the main feedstock properties. Analytical forms of these models are presented for gasoline aromatics and olefins and total conversion as well. The predictions of the models were satisfactory for all hydrocarbon groups. The models were also validated with experiments using two additional feedstocks in the pilot unit under a wide range of experimental conditions. © 1999 Elsevier Science B.V. All rights reserved."
result = classifier(example)
print(result)
# مثال على اختبار نص جديد
example = "Surgery-related infections have not been irradicated until now. To solve this problem, it is important to know the relationship between bacterial anatomy and bacterial behavior in the tug-of-war between host and pathogen. In this article, bacterial anatomy and functional behavior, host phagocytic activity, immune system, nutrition and antibiotics are reviewed to win the war against the tiny invaders and leave the host unharmed. My suggestion is that scientists should direct their studies not only to developing potent new antibiotics that will never give rise to drug-resistant mutants, but also to developing a very competitive immune system that can suppress or control infection without the aid of antibiotics."
result = classifier(example)
print(result)
# مثال على اختبار نص جديد
example = "Humans can naturally and effectively find salient regions in complex scenes. Motivated by this observation, attention mechanisms were introduced into computer vision with the aim of imitating this aspect of the human visual system. Such an attention mechanism can be regarded as a dynamic weight adjustment process based on features of the input image. Attention mechanisms have achieved great success in many visual tasks, including image classification, object detection, semantic segmentation, video understanding, image generation, 3D vision, multimodal tasks, and self-supervised learning. In this survey, we provide a comprehensive review of various attention mechanisms in computer vision and categorize them according to approach, such as channel attention, spatial attention, temporal attention, and branch attention; a related repository https://github.com/MenghaoGuo/Awesome-Vision-Attentions is dedicated to collecting related work. We also suggest future directions for attention mechanism research. [Figure not available: see fulltext.]"
result = classifier(example)
print(result)


Device set to use cpu


[{'label': 'psychology', 'score': 0.9966014623641968}]
[{'label': 'medicine', 'score': 0.9991180300712585}]
[{'label': 'computerscience', 'score': 0.9905416369438171}]
[{'label': 'chemistry', 'score': 0.9243378043174744}]
[{'label': 'biology', 'score': 0.9860060214996338}]
[{'label': 'computerscience', 'score': 0.8919618725776672}]
